# Train the Model

In [ ]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import grain.python as pygrain
import tiktoken
import optax # for optimizers

In [ ]:
from src.model.model import NanoLLM
from src.data.io import load_text_from_file
from src.data.processor import Processor
from src.config import TokenizerConfig, ModelConfig, TrainingConfig
from src.paths import DEFAULT_DATA_FILE, CHECKPOINTS_DIR

## Setup

### Understand characteristics of the dataset
This section uses methods for data loading and training that have been extracted to python modules. It does not use the fully automated training runner.

_The previous notebook is even more 'primitive', using early versions of the methods and classes._

In [ ]:
# Define delimiter and tokenizer name explicitly for TinyStories dataset
DELIMITER = "<|endoftext|>"
TOKENIZER_NAME = "gpt2"

# Set configs
model_config = ModelConfig()
tokenizer_config = TokenizerConfig(delimiter=DELIMITER, name=TOKENIZER_NAME)
training_config = TrainingConfig()

In [ ]:
# Load stories (includes validation of the path)
stories = load_text_from_file(
    file_path=DEFAULT_DATA_FILE, 
    delimiter=DELIMITER, 
    max_paragraphs=training_config.max_stories)

# Validate data characteristics (now included in the runner)
record_count = len(stories)
assert record_count != 0 # dataset should not be empty
batches_per_epoch = record_count // training_config.batch_size
assert batches_per_epoch > 0
print(f"Data characteristics: {record_count} stories or groupings of text; {batches_per_epoch} batches per epoch")

# Pre-process the data
# The dataloader groups individual samples into batches for the training loop to consume later.
processor = Processor(
    model_config=model_config,
    tokenizer_config=tokenizer_config,
    training_config=training_config,
)
dataloader = processor.process(stories)

In [ ]:
# Calculate training steps (warmup_steps has a floor of 1 to avoid a degenerate optax schedule on tiny datasets)
total_steps = batches_per_epoch * training_config.epochs
warmup_steps = max(1, int(total_steps * training_config.warmup_rate))
print(f"Total training steps: {total_steps:,}")
print(f"Warmup steps: {warmup_steps:,}")

### Define the training step: model, loss function, and learning rate schedule

In [ ]:
# Initialize model from config (untrained)
model = NanoLLM(model_config)

# loading from a fresh config instead of a checkpoint
previous_epochs_completed = 0 

In [ ]:
# Define loss function
def loss_fn(model, batch):
    inputs, targets = batch

    # Feed the inputs to the model. The results (logits) represent the denormalized likelihoods for each word in the dictionary that that word is next.
    logits = model(inputs)

    # Convert to probabilities (likely normalized)
    loss = optax.softmax_cross_entropy_with_integer_labels(
        logits, targets
    ).mean()

    # return both the probabilities and the original results
    return loss, logits

In [ ]:
# Create learning rate schedule using config values
learning_rate_schedule = optax.warmup_cosine_decay_schedule(
    init_value=training_config.lr_init_value,
    peak_value=training_config.lr_peak_value,
    warmup_steps=warmup_steps,
    decay_steps=total_steps,
    end_value=training_config.lr_end_value
)

### Define an optimizer to control how the network updates

In [ ]:
optimizer = nnx.ModelAndOptimizer(
    model,
    optax.adamw(
        learning_rate=learning_rate_schedule,
        weight_decay=training_config.weight_decay
    )
)

In [ ]:
# During training, we want to keep track of the average of the loss function: metrics.Average('loss')
metrics = nnx.MultiMetric(
    loss=nnx.metrics.Average('loss'),
)

### Define the training step method (JIT-compile)

In [ ]:
@nnx.jit
def train_step(model, optimizer, metrics, batch):
    grad_fn = nnx.value_and_grad(loss_fn, has_aux=True)
    (loss, logits), grads = grad_fn(model, batch)

    metrics.update(loss=loss, logits=logits, labels=batch[1])
    optimizer.update(grads)

## Run the training loop

In [ ]:
# NB: MetricsHistory has since been extracted as dataclass to improve control and access
metrics_history = {'train_loss': []}

# slides the inputs over by one index so we are always comparing the inputs to the next token (the target)
prep_target_batch = jax.vmap(
    lambda tokens: jnp.concatenate((tokens[1:], jnp.array([0]))))

for epoch in range(training_config.epochs):
    step = 0

    # pull a batch from the dataloader
    for batch in dataloader:
        input_batch = jnp.array(jnp.array(batch).T).astype(jnp.int32)
        target_batch = prep_target_batch(
            jnp.array(jnp.array(batch).T)).astype(jnp.int32)
        print(".", end="")

        # training step!
        train_step(model, optimizer, metrics, (input_batch, target_batch))

        if (step + 1) % training_config.log_every_n_steps == 0:
            for metric, value in metrics.compute().items():
                # update metrics history so we can track how well the model is doing
                metrics_history[f'train_{metric}'].append(value)
            metrics.reset()

            current_lr = learning_rate_schedule(step)
            print(f"\nEpoch: {epoch + 1}, Loss: {metrics_history['train_loss'][-1]:.4f}, "
                  f"LR: {current_lr:.2e}")
        step += 1

In [ ]:
# Visualize the training
import matplotlib.pyplot as plt
plt.plot(metrics_history['train_loss'])
plt.title('Training Loss')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.show()

## Save model checkpoints

In [ ]:
import dataclasses
from src.checkpoint import default_checkpoint_path, save_checkpoint, CheckpointMetadata
from src.paths import format_path_for_display

# Construct metadata to persist alongside orbax weights (for traceability of training parameters)
cumulative_epochs = previous_epochs_completed + training_config.epochs
final_loss = float(metrics_history['train_loss'][-1]) if metrics_history['train_loss'] else None # JAX's ArrayImpl isn't JSON-serializable, so it needs explicit conversion to float

metadata = CheckpointMetadata(
    cumulative_epochs_completed=cumulative_epochs,
    final_loss=final_loss, 
    model_config=dataclasses.asdict(model_config),
    training_config=dataclasses.asdict(training_config),
    tokenizer_config=dataclasses.asdict(tokenizer_config),
)

checkpoint_path = default_checkpoint_path()
save_checkpoint(model, checkpoint_path, metadata=metadata) # persists bundle of orbax weights and training metadata
print(f"Model saved as {format_path_for_display(checkpoint_path)}")

## [Alternative] Use CLI script to train a model from scratch
This script can be run without specifying any arguments. When no overrides are passed form the terminal as flag arguments, the script falls back to defaults for number of epochs and destination directory for persisted checkpoint.

_This is different from the script that loads pre-trained weights from a checkpoint to resume training._

In [ ]:
# shell magic allows us to run the CLI script directly from the notebook
# Use --epochs override to specify the number of subsequent training epochs.
!uv run nanollm-train --epochs 2